# Notebook 2 — Pipeline RAG

### Objectif
Construire le pipeline RAG complet :
- Découpage en chunks
- Vectorisation avec OpenAI Embeddings
- Stockage dans ChromaDB
- Retrieval et test de questions juridiques

### Prérequis
Avoir exécuté le notebook 01_chargement_donnees.ipynb

In [1]:
!pip install langchain langchain-openai langchain-community pypdf chromadb python-dotenv -q

## Étape 1 — Configuration et chargement des données

In [2]:
import os
from google.colab import userdata, drive
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
drive.mount('/content/drive', force_remount=True)

DOCS_DIR = "/content/drive/MyDrive/GEN_AI_Assistant_Juridique/data"
CHROMA_DIR = "/content/drive/MyDrive/GEN_AI_Assistant_Juridique/chroma_db"

# Chargement de tous les PDFs
pdf_files = list(Path(DOCS_DIR).glob("*.pdf"))
docs = []
for pdf_path in pdf_files:
    print(f"Chargement : {pdf_path.name}")
    loader = PyPDFLoader(str(pdf_path))
    docs.extend(loader.load())

print(f"\n✅ {len(docs)} pages chargées depuis {len(pdf_files)} fichiers")

Mounted at /content/drive
Chargement : Egalites_professionnelles.pdf
Chargement : Harcelement.pdf
Chargement : Contrat_de_travail.pdf
Chargement : Salaires_avantages.pdf
Chargement : Fin_de_contrat.pdf
Chargement : Syndicats.pdf
Chargement : sécurité_au_travail.pdf

✅ 479 pages chargées depuis 7 fichiers


## Étape 2 — Découpage en chunks

On teste 3 configurations différentes de chunking
pour comparer la qualité des résultats :

| Configuration | chunk_size | chunk_overlap |
|---|---|---|
| Small | 500 | 50 |
| Medium | 1000 | 150 |
| Large | 2000 | 200 |

Le chunk_size détermine la taille de chaque morceau.
Le chunk_overlap permet de garder du contexte
entre deux chunks consécutifs.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuration 1 — Small
splitter_small = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\nArticle", "\n\n", "\n", " "]
)

# Configuration 2 — Medium (notre choix principal)
splitter_medium = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\nArticle", "\n\n", "\n", " "]
)

# Configuration 3 — Large
splitter_large = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=["\nArticle", "\n\n", "\n", " "]
)

splits_small  = splitter_small.split_documents(docs)
splits_medium = splitter_medium.split_documents(docs)
splits_large  = splitter_large.split_documents(docs)

print("=== Comparaison des configurations ===")
print(f"Small  (500/50)   : {len(splits_small)} chunks")
print(f"Medium (1000/150) : {len(splits_medium)} chunks")
print(f"Large  (2000/200) : {len(splits_large)} chunks")

=== Comparaison des configurations ===
Small  (500/50)   : 4062 chunks
Medium (1000/150) : 2183 chunks
Large  (2000/200) : 1125 chunks


## Étape 3 — Analyse de la qualité des chunks

On analyse la taille moyenne des chunks
pour chaque configuration et on choisit
la meilleure pour notre cas d'usage juridique.

In [4]:
import numpy as np

configs = {
    "Small  (500/50) " : splits_small,
    "Medium (1000/150)": splits_medium,
    "Large  (2000/200)": splits_large
}

print("=== Analyse de la qualité des chunks ===\n")
for nom, splits in configs.items():
    longueurs = [len(s.page_content) for s in splits]
    print(f"--- {nom} ---")
    print(f"  Nombre de chunks  : {len(splits)}")
    print(f"  Longueur moyenne  : {int(np.mean(longueurs))} caractères")
    print(f"  Longueur minimale : {min(longueurs)} caractères")
    print(f"  Longueur maximale : {max(longueurs)} caractères")
    print(f"  Exemple de chunk  :")
    print(f"  {splits[0].page_content[:150]}...")
    print()

print("✅ Configuration choisie : Medium (1000/150)")
print("   Raison : équilibre optimal entre précision et contexte")
print("   pour des textes juridiques avec des articles structurés")

=== Analyse de la qualité des chunks ===

--- Small  (500/50)  ---
  Nombre de chunks  : 4062
  Longueur moyenne  : 432 caractères
  Longueur minimale : 22 caractères
  Longueur maximale : 500 caractères
  Exemple de chunk  :
  Partie législative - Première partie : Les relations individuelles de travail - Livre Ier : Dispositions préliminaires
Les dommages et intérêts répare...

--- Medium (1000/150) ---
  Nombre de chunks  : 2183
  Longueur moyenne  : 867 caractères
  Longueur minimale : 62 caractères
  Longueur maximale : 1000 caractères
  Exemple de chunk  :
  Partie législative - Première partie : Les relations individuelles de travail - Livre Ier : Dispositions préliminaires
Les dommages et intérêts répare...

--- Large  (2000/200) ---
  Nombre de chunks  : 1125
  Longueur moyenne  : 1607 caractères
  Longueur minimale : 118 caractères
  Longueur maximale : 2000 caractères
  Exemple de chunk  :
  Partie législative - Première partie : Les relations individuelles de travail - Livr

## Étape 4 — Vectorisation et stockage ChromaDB

On utilise OpenAIEmbeddings pour transformer
chaque chunk en vecteur numérique de 1536 dimensions.
Ces vecteurs sont stockés dans ChromaDB pour
permettre une recherche sémantique rapide.

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

print("Vectorisation en cours...")
embedding = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=splits_medium,
    embedding=embedding,
    persist_directory=CHROMA_DIR
)

print(f"✅ {len(splits_medium)} chunks vectorisés")
print(f"📁 Sauvegardé dans : {CHROMA_DIR}")
print(f"📊 Vecteurs stockés : {vectorstore._collection.count()}")

Vectorisation en cours...
✅ 2183 chunks vectorisés
📁 Sauvegardé dans : /content/drive/MyDrive/GEN_AI_Assistant_Juridique/chroma_db
📊 Vecteurs stockés : 2183


## Conclusion — Notebook 2
 3 configurations de chunking comparées :
   - Small  (500/50)  : 4062 chunks
   - Medium (1000/150): 2183 chunks
   - Large  (2000/200): 1125 chunks

Configuration Medium (1000/150) sélectionnée
2183 chunks vectorisés avec OpenAI Embeddings
Index ChromaDB sauvegardé sur Google Drive

### Pourquoi Medium ?
- Assez grand pour contenir un article complet
- Assez petit pour une recherche précise
- L'overlap de 150 caractères évite de perdre
  le contexte entre deux chunks
